In [1]:
import sys
from dataclasses import asdict
from datetime import datetime, timezone
from pathlib import Path

# Make src/ importable when the notebook runs from anywhere inside the project
root = Path.cwd()
while not (root / 'src').exists() and root != root.parent:
    root = root.parent
sys.path.insert(0, str(root / 'src'))

from pyspark.sql.types import StructType, StructField, StringType
from pipeline.session import get_spark_session
from pipeline.catalog import ensure_schema_exists, get_schema_fqn, list_tables, get_table_fqn, assert_table_exists
from pipeline.audit import AuditRun
from pipeline.errors import step
from pipeline.metadata import add_metadata
from pipeline.reader import read_table
from pipeline.validation import assert_columns_exist, count_duplicate_keys, count_null_business_keys
from pipeline.writer import insert_new_rows
from bronze.entity import EntityConfig

spark = get_spark_session('bronze_ingestion_test')

Picked up JAVA_TOOL_OPTIONS: -Djavax.net.ssl.trustStore=/home/charlescarpenter/.local/share/Project_Mercury/cacerts -Djavax.net.ssl.trustStorePassword=changeit
Picked up JAVA_TOOL_OPTIONS: -Djavax.net.ssl.trustStore=/home/charlescarpenter/.local/share/Project_Mercury/cacerts -Djavax.net.ssl.trustStorePassword=changeit
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/11 20:56:21 WARN Utils: Your hostname, DP41ZL4, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/10/11 20:56:21 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/charlescarpenter/venvs/mercury-delta/lib/python3.14/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/charlescarpenter/.ivy2.5.2/cache
The jars for the packages stored in: /home/charlescarpenter/.ivy2.5.2/jars
io.delta#delta-spark_4.1_2.13 added as a dep

In [2]:
import logging
from pipeline.events import EventBus
from pipeline.log import make_log_subscriber

logging.basicConfig(format='%(asctime)s %(levelname)s %(message)s', level=logging.INFO)

bus = EventBus()
bus.subscribe(make_log_subscriber(logging.getLogger('pipeline')))

1

In [3]:
COLUMNS = [
    'source_record_id', 'customer_id', 'first_name', 'last_name', 'email', 'phone',
    'date_of_birth', 'loyalty_status', 'marketing_opt_in', 'created_at', 'updated_at',
]
# Bronze keeps every source column as a string
SCHEMA = StructType([StructField(c, StringType(), True) for c in COLUMNS])

PROJECT_DATA = root / 'data' / 'B001'
CUSTOMER_FILE = PROJECT_DATA / 'customer_B001.csv'

CATALOG = 'forge'
SOURCE_SCHEMA = 'bronze'
TARGET_SCHEMA = 'silver_standardisation_test'


ensure_schema_exists(spark, CATALOG, SOURCE_SCHEMA)
ensure_schema_exists(spark, CATALOG, TARGET_SCHEMA)


print('source:', get_schema_fqn(CATALOG, SOURCE_SCHEMA))
print('target:', get_schema_fqn(CATALOG, TARGET_SCHEMA))


source: bronze
target: silver_standardisation_test


In [4]:
bronze_tables = list_tables(spark, CATALOG, SOURCE_SCHEMA)
print(bronze_tables)

['address', 'category', 'customer', 'order', 'order_item', 'payment', 'product', 'product_category', 'return', 'return_item']


In [5]:
entity_contracts = {
  "customer": {
    "business_key": ["customer_id"],
    "cast_rules": {}
  },
  "address": {
    "business_key": "address_id",
    "cast_rules": {}
  },
  "product": {
    "business_key": "product_id",
    "cast_rules": {
      "unit_price": "DECIMAL(18,2)"
    }
  },
  "category": {
    "business_key": "category_id",
    "cast_rules": {}
  },
  "product_category": {
    "business_key": "product_category_id",
    "cast_rules": {}
  },
  "order": {
    "business_key": "order_id",
    "cast_rules": {
      "order_timestamp": "TIMESTAMP",
      "subtotal_amount": "DECIMAL(18,2)",
      "discount_amount": "DECIMAL(18,2)",
      "tax_amount": "DECIMAL(18,2)",
      "shipping_amount": "DECIMAL(18,2)",
      "order_total_amount": "DECIMAL(18,2)"
    }
  },
  "order_item": {
    "business_key": "order_item_id",
    "cast_rules": {
      "quantity": "INT",
      "unit_price": "DECIMAL(18,2)",
      "discount_amount": "DECIMAL(18,2)",
      "tax_amount": "DECIMAL(18,2)",
      "line_total_amount": "DECIMAL(18,2)"
    }
  },
  "payment": {
    "business_key": "payment_id",
    "cast_rules": {
      "payment_timestamp": "TIMESTAMP",
      "amount": "DECIMAL(18,2)"
    }
  },
  "return": {
    "business_key": "return_id",
    "cast_rules": {
      "return_timestamp": "TIMESTAMP",
      "total_refund_amount": "DECIMAL(18,2)"
    }
  },
  "return_item": {
    "business_key": "return_item_id",
    "cast_rules": {
      "quantity": "INT",
      "refund_amount": "DECIMAL(18,2)"
    }
  }
}


In [6]:
from typing import Mapping

COUNTRY_ALIASES: Mapping[str, str] = {   # keys already UPPER + trimmed
    "GB": "GB", "UK": "GB", "GBR": "GB", "GREAT BRITAIN": "GB",
    "UNITED KINGDOM": "GB", "ENGLAND": "GB", "SCOTLAND": "GB",
    "WALES": "GB", "NORTHERN IRELAND": "GB"}
BOOLEAN_TRUE:  tuple[str, ...] = ("Y", "YES", "TRUE", "T", "1")
BOOLEAN_FALSE: tuple[str, ...] = ("N", "NO", "FALSE", "F", "0")
DATE_FORMATS: tuple[str, ...] = ("yyyy-MM-dd", "dd-MM-yyyy", "yyyy/MM/dd", "dd/MM/yyyy")
UK_PHONE = {"country_code": "44", "trunk_prefix": "0", "national_lengths": (10,)}
UK_SUBDIVISION_MAP: Mapping[str, str] = {   # lower-cased region -> UK nation
    "scotland": "Scotland", "wales": "Wales", "northern ireland": "Northern Ireland",
    "greater london": "England", "merseyside": "England", "bristol": "England",
    "west midlands": "England", "south yorkshire": "England",
    "west yorkshire": "England", "north yorkshire": "England",
    "greater manchester": "England", "nottinghamshire": "England"}



In [7]:

CATALOG = "spark_catalog"
BRONZE_SCHEMA, STAGE_SCHEMA = "bronze", "silver_stage"
TRUSTED_SCHEMA, QUARANTINE_SCHEMA, AUDIT_SCHEMA_NAME = "silver", "quarantine", "audit"
STANDARDISATION_AUDIT_TABLE = f"{CATALOG}.{AUDIT_SCHEMA_NAME}.silver_standardisation_audit"
DQ_AUDIT_TABLE = f"{CATALOG}.{AUDIT_SCHEMA_NAME}.silver_dq_audit"
NULL_TOKENS: tuple[str, ...] = ("", "NULL", "null", "N/A", "NA", "None")
RAW_SUFFIX = "_raw"
STATUS_SUCCESS, STATUS_FAILED = "SUCCESS", "FAILED"
DQ_HARD_COLUMN, DQ_SOFT_COLUMN = "_dq_hard_failed_rule_ids", "_dq_soft_failed_rule_ids"
ORDER_BY: tuple[str, ...] = ("updated_at", "_ingested_timestamp")   # dedupe winner = largest, in order
DECIMAL_TYPE = "decimal(18,2)"


In [8]:
from collections.abc import Mapping

TIMESTAMP_COLUMNS: tuple[str, ...] = ("created_at", "updated_at")   # cast to TIMESTAMP for every entity


def string_rules(columns):
    """Default rule (trim and NULL tokens only) for each listed column."""
    return {column: "string" for column in columns}


BUSINESS_KEYS: Mapping[str, tuple[str, ...]] = {
    "address": ("address_id",),
    "category": ("category_id",),
    "customer": ("customer_id",),
    "order": ("order_id",),
    "order_item": ("order_item_id",),
    "payment": ("payment_id",),
    "product": ("product_id",),
    "product_category": ("product_category_id",),
    "return": ("return_id",),
    "return_item": ("return_item_id",),
}

# entity -> {column: rule}. Casts (int, decimal, timestamp) live in entity_contracts, not here.
# Columns not listed here are passed through untouched, so every data column is listed with "string" as the minimum.
COLUMN_RULES: Mapping[str, Mapping[str, str]] = {
    "address": {
        **string_rules(["source_record_id", "address_id", "customer_id"]),
        "address_type": "upper",
        "address_line_1": "title",
        "address_line_2": "title",
        "city": "title",
        "state_region": "title",
        "postal_code": "upper",
        "country_code": "country",
        "is_primary": "boolean",
    },
    "category": {
        **string_rules(["source_record_id", "category_id", "parent_category_id"]),
        "category_name": "title",
        "category_code": "upper",
        "is_active": "boolean",
    },
    "customer": {
        **string_rules(["source_record_id", "customer_id"]),
        "first_name": "title",
        "last_name": "title",
        "email": "lower",
        "phone": "phone",
        "date_of_birth": "date",
        "loyalty_status": "upper",
        "marketing_opt_in": "boolean",
    },
    "order": {
        **string_rules(
            ["source_record_id", "order_id", "customer_id", "billing_address_id", "shipping_address_id"]
        ),
        "channel": "upper",
        "order_status": "upper",
        "currency_code": "upper",
    },
    "order_item": {
        **string_rules(["source_record_id", "order_item_id", "order_id", "product_id"]),
    },
    "payment": {
        **string_rules(["source_record_id", "payment_id", "order_id", "customer_id", "transaction_reference"]),
        "payment_method": "upper",
        "payment_status": "upper",
        "currency_code": "upper",
    },
    "product": {
        **string_rules(["source_record_id", "product_id", "product_name", "brand"]),
        "sku": "upper",
        "currency_code": "upper",
        "is_active": "boolean",
    },
    "product_category": {
        **string_rules(["source_record_id", "product_category_id", "product_id", "category_id"]),
        "is_primary": "boolean",
        "active_from": "date",
        "active_to": "date",
    },
    "return": {
        **string_rules(["source_record_id", "return_id", "order_id", "customer_id"]),
        "return_status": "upper",
        "return_reason": "upper",
    },
    "return_item": {
        **string_rules(["source_record_id", "return_item_id", "return_id", "order_item_id", "product_id"]),
        "return_reason": "upper",
    },
}

# entity -> {new_column: (source_column, rule)}
DERIVED_COLUMN_RULES: Mapping[str, Mapping[str, tuple[str, str]]] = {
    "address": {"uk_nation": ("state_region", "uk_subdivision")},
}


In [9]:
from pipeline.metadata import add_record_hash
from pipeline.writer import replace_table
from silver.curation import add_lineage_columns, deduplicate, source_columns

SOURCE_SYSTEM = "OLTP"
from silver.contract import EntityContract
from silver.standardisation.casting import cast_columns
from silver.standardisation.column_rules import StandardisationConfig, apply_column_rules, build_rule_registry

STANDARDISATION_CONFIG = StandardisationConfig(
    null_tokens=NULL_TOKENS,
    boolean_true=BOOLEAN_TRUE,
    boolean_false=BOOLEAN_FALSE,
    date_formats=DATE_FORMATS,
    country_aliases=COUNTRY_ALIASES,
    phone=UK_PHONE,
    uk_subdivision_map=UK_SUBDIVISION_MAP,
)
# Rules registered so far. date, phone, country and uk_subdivision are not built yet, so a contract naming them fails to load.
REGISTRY = build_rule_registry(STANDARDISATION_CONFIG)


In [10]:
TIMESTAMP_CAST_RULES = {column: "TIMESTAMP" for column in TIMESTAMP_COLUMNS}

contracts = {}
for entity, entity_contract_dict in entity_contracts.items():
    source_table_fqn = get_table_fqn(CATALOG, SOURCE_SCHEMA, entity)
    assert_table_exists(spark, source_table_fqn)
    target_table_fqn = get_table_fqn(CATALOG, TARGET_SCHEMA, entity)
    contract_dict = {
        **entity_contract_dict,
        "cast_rules": {**TIMESTAMP_CAST_RULES, **entity_contract_dict.get("cast_rules", {})},
    }
    contracts[entity] = EntityContract.from_dict(
        entity, contract_dict, COLUMN_RULES.get(entity, {}), source_table_fqn, target_table_fqn, valid_rules=REGISTRY
    )


In [11]:
for entity, entity_contract in contracts.items():
    print("-" * 40)
    print(f"Entity: {entity}")
    print(f"Business Key: {entity_contract.business_key}")
    print(f"Source Table: {entity_contract.source_table_fqn}")
    print(f"Target Table: {entity_contract.target_table_fqn}")
    for rule in entity_contract.cast_rules:
        print(f"Column: {rule.column_name}, Data Type: {rule.data_type}")

----------------------------------------
Entity: customer
Business Key: ('customer_id',)
Source Table: bronze.customer
Target Table: silver_standardisation_test.customer
Column: created_at, Data Type: TIMESTAMP
Column: updated_at, Data Type: TIMESTAMP
----------------------------------------
Entity: address
Business Key: ('address_id',)
Source Table: bronze.address
Target Table: silver_standardisation_test.address
Column: created_at, Data Type: TIMESTAMP
Column: updated_at, Data Type: TIMESTAMP
----------------------------------------
Entity: product
Business Key: ('product_id',)
Source Table: bronze.product
Target Table: silver_standardisation_test.product
Column: created_at, Data Type: TIMESTAMP
Column: updated_at, Data Type: TIMESTAMP
Column: unit_price, Data Type: DECIMAL(18,2)
----------------------------------------
Entity: category
Business Key: ('category_id',)
Source Table: bronze.category
Target Table: silver_standardisation_test.category
Column: created_at, Data Type: TIMEST

In [12]:
ensure_schema_exists(spark, CATALOG, STAGE_SCHEMA)

records = []
def standardise_silver_table(contract: EntityContract) -> None:
    stage_table = get_table_fqn(CATALOG, STAGE_SCHEMA, contract.entity)
    with AuditRun(
        "silver_standardisation", contract.entity, stage_table, records.append, bus=bus
    ) as run:
        with step("load_bronze_data", bus=bus):
            assert_table_exists(spark, contract.source_table_fqn)
            df = read_table(spark, contract.source_table_fqn)
            run.add("table", contract.source_table_fqn)
            run.add("source_row_count", df.count())

        with step("add_lineage_columns", bus=bus):
            # Before standardising, so _raw_record_payload holds the original values
            df = add_lineage_columns(df, source_system=SOURCE_SYSTEM, pipeline_run_id=run.record.run_id)

        with step("cast", bus=bus):
            cast = cast_columns(df, contract.cast_rules, NULL_TOKENS)
            df = cast.df
            for name, value in cast.invalid_counts.items():
                run.add(name, value)

        with step("standardise", bus=bus):
            standardised = apply_column_rules(
                df, contract.column_rules, REGISTRY, NULL_TOKENS, DERIVED_COLUMN_RULES.get(contract.entity)
            )
            df = standardised.df
            for name, value in standardised.invalid_counts.items():
                run.add(name, value)

        with step("deduplicate", bus=bus):
            # Hash the standardised values, so exact duplicates are caught after standardising
            df = add_record_hash(df, source_columns(df), column="_record_hash")
            df, dedupe_counts = deduplicate(df, contract.business_key)
            for name, value in dedupe_counts.items():
                run.add(name, value)

        with step("write_silver_stage", bus=bus):
            replace_table(df, stage_table)
            run.add("stage_row_count", spark.read.table(stage_table).count())


In [13]:
for contract in contracts.values():
    try:
        standardise_silver_table(contract)
    except Exception as exc:
        print(f"{contract.entity}: failed: {exc}")

for record in records:
    print(record.entity, record.status, record.metrics, record.error_message or "")


2026-10-11 20:56:33,543 INFO silver_standardisation customer run e995dff1-e836-4788-bfbd-dbd9fb029473 started
2026-10-11 20:56:39,330 INFO step 'load_bronze_data' completed in 5.78s         
2026-10-11 20:56:39,449 INFO step 'add_lineage_columns' completed in 0.12s
2026-10-11 20:56:43,670 INFO step 'cast' completed in 4.22s                     
2026-10-11 20:56:45,554 INFO step 'standardise' completed in 1.88s
2026-10-11 20:56:50,097 INFO step 'deduplicate' completed in 4.54s
2026-10-11 20:56:55,303 INFO step 'write_silver_stage' completed in 5.20s       
2026-10-11 20:56:55,305 INFO silver_standardisation customer run e995dff1-e836-4788-bfbd-dbd9fb029473 succeeded: {'table': 'bronze.customer', 'source_row_count': 50, 'invalid_value_count': 0, 'null_business_key_removed_count': 0, 'duplicate_hash_removed_count': 0, 'superseded_removed_count': 0, 'stage_row_count': 50}
2026-10-11 20:56:55,307 INFO silver_standardisation address run de3f301e-43ce-405a-8d99-fb960d489b22 started
2026-10-11

customer success {'table': 'bronze.customer', 'source_row_count': 50, 'invalid_value_count': 0, 'null_business_key_removed_count': 0, 'duplicate_hash_removed_count': 0, 'superseded_removed_count': 0, 'stage_row_count': 50} 
address success {'table': 'bronze.address', 'source_row_count': 68, 'invalid_value_count': 0, 'null_business_key_removed_count': 0, 'duplicate_hash_removed_count': 0, 'superseded_removed_count': 0, 'stage_row_count': 68} 
product success {'table': 'bronze.product', 'source_row_count': 200, 'invalid_value_count': 0, 'null_business_key_removed_count': 0, 'duplicate_hash_removed_count': 0, 'superseded_removed_count': 0, 'stage_row_count': 200} 
category success {'table': 'bronze.category', 'source_row_count': 6, 'invalid_value_count': 0, 'null_business_key_removed_count': 0, 'duplicate_hash_removed_count': 0, 'superseded_removed_count': 0, 'stage_row_count': 6} 
product_category success {'table': 'bronze.product_category', 'source_row_count': 310, 'invalid_value_count'

In [19]:
from IPython.display import display

from pipeline.catalog import list_tables

tables = list_tables(spark, "forge", "silver_standardisation_test")
print(tables)

silver_stage_address = read_table(spark, "silver_stage.product_category")
display(silver_stage_address.toPandas())

[]


,source_record_id,product_category_id,product_id,category_id,is_primary,active_from,active_to,created_at,updated_at,_source_file,_record_hash,_run_id,_ingested_at,_raw_record_payload,_source_system,_source_file_path,_source_file_name,_source_batch_id,_pipeline_run_id,_ingested_timestamp
0,SRC_PRODUCT_CATEGORY_B001_000000001,PC00000001,P000001,CAT0006,True,2026-04-19,None,2026-04-19 05:32:32,2026-04-19 05:32:32,file:/home/charlescarpenter/python/spark_local...,448799710e036d8d73413c87a5d2148de6ccb4392107aa...,3bdca82d-859e-4429-bdff-30ce3a037880,2026-10-11 14:12:58.389669,"{""source_record_id"":""SRC_PRODUCT_CATEGORY_B001...",OLTP,file:/home/charlescarpenter/python/spark_local...,product_category_B001.csv,3bdca82d-859e-4429-bdff-30ce3a037880,f2015d34-5e06-4ea3-982e-ddfdb451ce55,2026-10-11 19:57:23.330857
1,SRC_PRODUCT_CATEGORY_B001_000000002,PC00000002,P000001,CAT0005,False,2026-04-19,None,2026-04-19 05:32:32,2026-04-19 05:32:32,file:/home/charlescarpenter/python/spark_local...,f417ec7243ecca59e286e712f8387b11fffd96436f276d...,3bdca82d-859e-4429-bdff-30ce3a037880,2026-10-11 14:12:58.389669,"{""source_record_id"":""SRC_PRODUCT_CATEGORY_B001...",OLTP,file:/home/charlescarpenter/python/spark_local...,product_category_B001.csv,3bdca82d-859e-4429-bdff-30ce3a037880,f2015d34-5e06-4ea3-982e-ddfdb451ce55,2026-10-11 19:57:23.330857
2,SRC_PRODUCT_CATEGORY_B001_000000003,PC00000003,P000001,CAT0004,False,2026-04-19,None,2026-04-19 05:32:32,2026-04-19 05:32:32,file:/home/charlescarpenter/python/spark_local...,90d2a795a81afd0e66ab782761f6a8da67e64a7cc192b9...,3bdca82d-859e-4429-bdff-30ce3a037880,2026-10-11 14:12:58.389669,"{""source_record_id"":""SRC_PRODUCT_CATEGORY_B001...",OLTP,file:/home/charlescarpenter/python/spark_local...,product_category_B001.csv,3bdca82d-859e-4429-bdff-30ce3a037880,f2015d34-5e06-4ea3-982e-ddfdb451ce55,2026-10-11 19:57:23.330857
3,SRC_PRODUCT_CATEGORY_B001_000000004,PC00000004,P000002,CAT0005,True,2024-03-22,None,2024-03-22 12:49:21,2024-03-22 12:49:21,file:/home/charlescarpenter/python/spark_local...,5ba28c9a28f0770a013bec3327f60b15d39024035acbe7...,3bdca82d-859e-4429-bdff-30ce3a037880,2026-10-11 14:12:58.389669,"{""source_record_id"":""SRC_PRODUCT_CATEGORY_B001...",OLTP,file:/home/charlescarpenter/python/spark_local...,product_category_B001.csv,3bdca82d-859e-4429-bdff-30ce3a037880,f2015d34-5e06-4ea3-982e-ddfdb451ce55,2026-10-11 19:57:23.330857
4,SRC_PRODUCT_CATEGORY_B001_000000005,PC00000005,P000003,CAT0004,True,2023-03-07,None,2023-03-07 12:56:57,2023-03-07 12:56:57,file:/home/charlescarpenter/python/spark_local...,10699c54e50c13e6b8538faa60b8f15e39dc81ad1fcd5c...,3bdca82d-859e-4429-bdff-30ce3a037880,2026-10-11 14:12:58.389669,"{""source_record_id"":""SRC_PRODUCT_CATEGORY_B001...",OLTP,file:/home/charlescarpenter/python/spark_local...,product_category_B001.csv,3bdca82d-859e-4429-bdff-30ce3a037880,f2015d34-5e06-4ea3-982e-ddfdb451ce55,2026-10-11 19:57:23.330857
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
305,SRC_PRODUCT_CATEGORY_B001_000000306,PC00000306,P000197,CAT0003,True,2023-06-05,None,2023-06-05 12:50:01,2023-06-05 12:50:01,file:/home/charlescarpenter/python/spark_local...,f15881b0250cc79a79667ae9f9d182a9f1ab19b8e408cd...,3bdca82d-859e-4429-bdff-30ce3a037880,2026-10-11 14:12:58.389669,"{""source_record_id"":""SRC_PRODUCT_CATEGORY_B001...",OLTP,file:/home/charlescarpenter/python/spark_local...,product_category_B001.csv,3bdca82d-859e-4429-bdff-30ce3a037880,f2015d34-5e06-4ea3-982e-ddfdb451ce55,2026-10-11 19:57:23.330857
306,SRC_PRODUCT_CATEGORY_B001_000000307,PC00000307,P000197,CAT0005,False,2023-06-05,None,2023-06-05 12:50:01,2023-06-05 12:50:01,file:/home/charlescarpenter/python/spark_local...,6d1ef19e1f94f86e08befc35d710f22d085c3aee2a3e76...,3bdca82d-859e-4429-bdff-30ce3a037880,2026-10-11 14:12:58.389669,"{""source_record_id"":""SRC_PRODUCT_CATEGORY_B001...",OLTP,file:/home/charlescarpenter/python/spark_local...,product_category_B001.csv,3bdca82d-859e-4429-bdff-30ce3a037

In [15]:
# spark.sql(f'DROP SCHEMA IF EXISTS {get_schema_fqn(CATALOG, TARGET_SCHEMA)} CASCADE')